In [1]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive has been mounted successfully.")

Mounted at /content/drive
Google Drive has been mounted successfully.


In [2]:
import os

# Define the directory where the XML files are located
# List all files and directories within the specified path
files_in_mydrive = os.listdir('/content/drive/MyDrive/Biorad/xml_files')

# Print the list of files to verify contents
for item in files_in_mydrive:
    print(item)

591-ws-training.xml
563-ws-training.xml
588-ws-training.xml
559-ws-testing.xml
563-ws-testing.xml
588-ws-testing.xml
570-ws-testing.xml
575-ws-training.xml
570-ws-training.xml
575-ws-testing.xml
559-ws-training.xml
591-ws-testing.xml


In [3]:
#1. Load XML and inspect top-level structure
import xml.etree.ElementTree as ET

xml_path = "/content/drive/MyDrive/Biorad/xml_files/591-ws-training.xml"

tree = ET.parse(xml_path)
root = tree.getroot()

print("Root tag:", root.tag)
print("Patient attributes:", root.attrib)

print("\nTop level sections:")
for child in root:
    print(child.tag)

Root tag: patient
Patient attributes: {'id': '591', 'weight': '99', 'insulin_type': 'Novalog'}

Top level sections:
glucose_level
finger_stick
basal
temp_basal
bolus
meal
sleep
work
stressors
hypo_event
illness
exercise
basis_heart_rate
basis_gsr
basis_skin_temperature
basis_air_temperature
basis_steps
basis_sleep


In [4]:
#2. Extract all attribute names used in events
from collections import defaultdict

tag_attributes = defaultdict(set)

for section in root:

    for event in section.findall("event"):

        for attr in event.attrib:
            tag_attributes[section.tag].add(attr)

print("\nAttributes used in each section:\n")

for tag, attrs in tag_attributes.items():
    print(tag, "->", sorted(list(attrs)))


Attributes used in each section:

glucose_level -> ['ts', 'value']
finger_stick -> ['ts', 'value']
basal -> ['ts', 'value']
temp_basal -> ['ts_begin', 'ts_end', 'value']
bolus -> ['bwz_carb_input', 'dose', 'ts_begin', 'ts_end', 'type']
meal -> ['carbs', 'ts', 'type']
sleep -> ['quality', 'ts_begin', 'ts_end']
hypo_event -> ['ts']
illness -> ['description', 'ts_begin', 'ts_end', 'type']
exercise -> ['competitive', 'duration', 'intensity', 'ts', 'type']
basis_heart_rate -> ['ts', 'value']
basis_gsr -> ['ts', 'value']
basis_skin_temperature -> ['ts', 'value']
basis_air_temperature -> ['ts', 'value']
basis_steps -> ['ts', 'value']
basis_sleep -> ['quality', 'tbegin', 'tend', 'type']


In [5]:
#3. Count number of events in each section
print("\nEvent counts per section:\n")

for section in root:
    events = section.findall("event")
    print(section.tag, ":", len(events))


Event counts per section:

glucose_level : 10847
finger_stick : 427
basal : 113
temp_basal : 33
bolus : 261
meal : 212
sleep : 44
work : 0
stressors : 0
hypo_event : 7
illness : 1
exercise : 23
basis_heart_rate : 12276
basis_gsr : 12079
basis_skin_temperature : 12086
basis_air_temperature : 12086
basis_steps : 12054
basis_sleep : 964


In [6]:
#4. Detect timestamp fields used
timestamp_fields = set()

for section in root:
    for event in section.findall("event"):
        for attr in event.attrib:
            if "ts" in attr:
                timestamp_fields.add(attr)

print("Timestamp fields found:", timestamp_fields)

Timestamp fields found: {'ts', 'ts_end', 'ts_begin'}


In [7]:
#5. Collect all unique attributes globally
all_attributes = set()

for section in root:
    for event in section.findall("event"):
        all_attributes.update(event.attrib.keys())

print("\nAll event attributes found:\n")
print(sorted(all_attributes))


All event attributes found:

['bwz_carb_input', 'carbs', 'competitive', 'description', 'dose', 'duration', 'intensity', 'quality', 'tbegin', 'tend', 'ts', 'ts_begin', 'ts_end', 'type', 'value']


In [8]:
#6. Print a few example events for each section
for section in root:

    print("\nSECTION:", section.tag)

    for i, event in enumerate(section.findall("event")[:3]):
        print(event.attrib)


SECTION: glucose_level
{'ts': '30-11-2021 17:06:00', 'value': '160'}
{'ts': '30-11-2021 17:11:00', 'value': '158'}
{'ts': '30-11-2021 17:16:00', 'value': '160'}

SECTION: finger_stick
{'ts': '30-11-2021 16:42:11', 'value': '187'}
{'ts': '30-11-2021 16:55:21', 'value': '163'}
{'ts': '30-11-2021 17:57:05', 'value': '263'}

SECTION: basal
{'ts': '30-11-2021 00:00:00', 'value': '1.05'}
{'ts': '30-11-2021 03:00:00', 'value': '1.25'}
{'ts': '30-11-2021 07:30:00', 'value': '0.9'}

SECTION: temp_basal
{'ts_begin': '01-12-2021 09:01:46', 'ts_end': '01-12-2021 10:31:46', 'value': '0.27'}
{'ts_begin': '05-12-2021 09:21:44', 'ts_end': '05-12-2021 10:00:09', 'value': '0.0'}
{'ts_begin': '08-12-2021 08:42:34', 'ts_end': '08-12-2021 09:12:26', 'value': '0.0'}

SECTION: bolus
{'ts_begin': '30-11-2021 16:42:11', 'ts_end': '30-11-2021 16:42:11', 'type': 'normal', 'dose': '4.3', 'bwz_carb_input': '30'}
{'ts_begin': '30-11-2021 17:57:05', 'ts_end': '30-11-2021 17:57:05', 'type': 'normal', 'dose': '0.3', 

In [9]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np


STANDARD_COLUMNS = [
    "patient_id","timestamp",
    "glucose_level","finger_stick","basal","temp_basal",
    "bolus_dose","bolus_carb_input","bolus_type",
    "meal_type","meal_carbs",
    "exercise_intensity","exercise_duration","exercise_competitive",
    "sleep_quality","basis_sleep_quality",
    "hypo_event",
    "illness_type","illness_description",
    "basis_heart_rate","basis_gsr","basis_skin_temperature",
    "basis_air_temperature","basis_steps"
]


def clean(v):
    if v is None:
        return np.nan
    v = str(v).strip()
    if v == "":
        return np.nan
    return v


def parse_time(t):
    if t is None or t == "":
        return None
    return pd.to_datetime(t, format="%d-%m-%Y %H:%M:%S")


def expand_interval(start, end):
    """expand interval only if gap >5 minutes"""
    times = [start]
    cur = start

    while True:
        nxt = cur + pd.Timedelta(minutes=5)
        if nxt >= end:
            break
        times.append(nxt)
        cur = nxt

    times.append(end)
    return times


def parse_diabetes_xml(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    patient_id = root.attrib.get("id")

    rows = {}

    def get_row(ts):
        if ts not in rows:
            rows[ts] = {c: np.nan for c in STANDARD_COLUMNS}
            rows[ts]["timestamp"] = ts
            rows[ts]["patient_id"] = patient_id
        return rows[ts]


    for section in root:

        tag = section.tag

        for event in section.findall("event"):

            a = event.attrib

            ts = parse_time(a.get("ts") or a.get("ts_begin") or a.get("tbegin"))
            ts_end = parse_time(a.get("ts_end") or a.get("tend"))

            if ts is None:
                continue

            if ts_end is not None and ts_end < ts:
                ts, ts_end = ts_end, ts

            # ---------- GLUCOSE ----------
            if tag == "glucose_level":

                val = clean(a.get("value"))
                if val is None:
                    continue

                row = get_row(ts)
                row["glucose_level"] = float(val)


            # ---------- FINGER STICK ----------
            elif tag == "finger_stick":

                row = get_row(ts)
                row["finger_stick"] = float(clean(a.get("value")))


            # ---------- BASAL ----------
            elif tag == "basal":

                row = get_row(ts)
                row["basal"] = float(clean(a.get("value")))


            # ---------- TEMP BASAL ----------
            elif tag == "temp_basal":

                if ts_end is None:
                    ts_end = ts

                times = expand_interval(ts, ts_end)

                val = float(clean(a.get("value")))

                for t in times:
                    row = get_row(t)
                    row["temp_basal"] = val   # overwrite allowed


            # ---------- BOLUS ----------
            elif tag == "bolus":

                row = get_row(ts)

                row["bolus_dose"] = float(clean(a.get("dose")))
                row["bolus_carb_input"] = float(clean(a.get("bwz_carb_input")))
                row["bolus_type"] = clean(a.get("type"))


            # ---------- MEAL ----------
            elif tag == "meal":

                row = get_row(ts)

                row["meal_type"] = clean(a.get("type"))
                row["meal_carbs"] = float(clean(a.get("carbs")))


            # ---------- EXERCISE ----------
            elif tag == "exercise":

                intensity = float(clean(a.get("intensity")))
                duration = clean(a.get("duration"))

                if duration is not None:
                    ts_end = ts + pd.Timedelta(minutes=float(duration))
                else:
                    ts_end = ts

                times = expand_interval(ts, ts_end)

                for t in times:
                    row = get_row(t)
                    row["exercise_intensity"] = intensity
                    row["exercise_duration"] = float(duration)
                    row["exercise_competitive"] = clean(a.get("competitive"))


            # ---------- SLEEP ----------
            elif tag == "sleep":

                if ts_end is None:
                    ts_end = ts

                times = expand_interval(ts, ts_end)

                quality = float(clean(a.get("quality")))

                for t in times:
                    row = get_row(t)
                    row["sleep_quality"] = quality


            # ---------- BASIS SLEEP ----------
            elif tag == "basis_sleep":

                if ts_end is None:
                    ts_end = ts

                times = expand_interval(ts, ts_end)

                quality = float(clean(a.get("quality")))

                for t in times:
                    row = get_row(t)
                    row["basis_sleep_quality"] = quality


            # ---------- HYPO ----------
            elif tag == "hypo_event":

                row = get_row(ts)
                row["hypo_event"] = 1


            # ---------- ILLNESS ----------
            elif tag == "illness":

                if ts_end is None:
                    ts_end = ts + pd.Timedelta(days=1)

                times = expand_interval(ts, ts_end)

                for t in times:
                    row = get_row(t)
                    row["illness_type"] = clean(a.get("type"))
                    row["illness_description"] = clean(a.get("description"))


            # ---------- BASIS SENSORS ----------
            elif tag == "basis_heart_rate":

                row = get_row(ts)
                row["basis_heart_rate"] = float(clean(a.get("value")))


            elif tag == "basis_gsr":

                row = get_row(ts)
                row["basis_gsr"] = float(clean(a.get("value")))


            elif tag == "basis_skin_temperature":

                row = get_row(ts)
                row["basis_skin_temperature"] = float(clean(a.get("value")))


            elif tag == "basis_air_temperature":

                row = get_row(ts)
                row["basis_air_temperature"] = float(clean(a.get("value")))


            elif tag == "basis_steps":

                row = get_row(ts)
                row["basis_steps"] = float(clean(a.get("value")))


    df = pd.DataFrame(rows.values())

    df = df.sort_values("timestamp").reset_index(drop=True)

    # basal persistence
    if "basal" in df.columns:
        df["basal"] = df["basal"].ffill()

    return df

In [10]:
import os

folder = "/content/drive/MyDrive/Biorad/xml_files"

patient_dfs = {}

for file in os.listdir(folder):

    if file.endswith(".xml"):

        path = os.path.join(folder, file)

        df = parse_diabetes_xml(path)

        pid = df["patient_id"].iloc[0]

        if pid not in patient_dfs:
            patient_dfs[pid] = []

        patient_dfs[pid].append(df)


# combine training + testing per patient
for pid in patient_dfs:
    patient_dfs[pid] = pd.concat(patient_dfs[pid]).sort_values("timestamp").reset_index(drop=True)


print("Patients loaded:", patient_dfs.keys())

Patients loaded: dict_keys(['591', '563', '588', '559', '570', '575'])


In [12]:
import pandas as pd
import plotly.graph_objects as go

patient_id = "591"

pdf = df[df["patient_id"] == patient_id].copy()
pdf["timestamp"] = pd.to_datetime(pdf["timestamp"])

continuous_attrs = [
    "basis_heart_rate",
    "basis_steps",
    "basis_gsr",
    "basis_skin_temperature",
    "basis_air_temperature",
    "basis_sleep_quality"
]

event_attrs = [
    "meal_type",
    "exercise_intensity",
    "hypo_event",
    "finger_stick",
    "bolus_dose",
    "basal",
    "temp_basal"
]

# ensure numeric columns
pdf["glucose_level"] = pd.to_numeric(pdf["glucose_level"], errors="coerce")
for attr in continuous_attrs:
    pdf[attr] = pd.to_numeric(pdf[attr], errors="coerce")

fig = go.Figure()

# Glucose trace
fig.add_trace(go.Scatter(
    x=pdf["timestamp"],
    y=pdf["glucose_level"],
    mode="lines+markers",
    connectgaps=True,
    line=dict(width=2),
    marker=dict(size=4),
    name="Glucose",
    yaxis="y1"
))

# Continuous attributes on secondary axis
for attr in continuous_attrs:
    fig.add_trace(go.Scatter(
        x=pdf["timestamp"],
        y=pdf[attr],
        mode="lines+markers",
        connectgaps=True,
        line=dict(width=2),
        marker=dict(size=4),
        name=attr,
        visible=False,
        yaxis="y2"
    ))

buttons = []

# NONE option
buttons.append(dict(
    label="None",
    method="update",
    args=[{"visible":[True]+[False]*len(continuous_attrs)}, {"shapes": []}]
))

# Continuous attribute dropdown
for i, attr in enumerate(continuous_attrs):

    vis = [True] + [False]*len(continuous_attrs)
    vis[i+1] = True

    buttons.append(dict(
        label=attr,
        method="update",
        args=[{"visible": vis}, {"shapes": []}]
    ))

# Event dropdown
for event in event_attrs:

    shapes = []
    events = pdf[pdf[event].notna()]

    for _, r in events.iterrows():
        shapes.append(dict(
            type="line",
            x0=r["timestamp"],
            x1=r["timestamp"],
            y0=0,
            y1=r["glucose_level"],
            line=dict(color="red", dash="dot", width=2)
        ))

    buttons.append(dict(
        label=event,
        method="update",
        args=[{"visible":[True]+[False]*len(continuous_attrs)}, {"shapes": shapes}]
    ))

# Date dropdown based on available dataset dates
unique_dates = sorted(pdf["timestamp"].dt.date.unique())

date_buttons = []

for d in unique_dates:
    start = pd.Timestamp(d)
    end = start + pd.Timedelta(days=1)

    date_buttons.append(dict(
        label=str(d),
        method="relayout",
        args=[{"xaxis.range":[start,end]}]
    ))

fig.update_layout(

    title="Glucose vs Selected Attribute",

    xaxis=dict(
        title="Timestamp",
        type="date",

        rangeselector=dict(
            buttons=[
                dict(count=1, label="1h", step="hour", stepmode="backward"),
                dict(count=6, label="6h", step="hour", stepmode="backward"),
                dict(count=12, label="12h", step="hour", stepmode="backward"),
                dict(count=1, label="1d", step="day", stepmode="backward"),
                dict(count=7, label="7d", step="day", stepmode="backward"),
                dict(step="all")
            ]
        ),

        rangeslider=dict(visible=True)
    ),

    yaxis=dict(
        title="Glucose (mg/dL)",
        range=[0,400],
        type="linear"
    ),

    yaxis2=dict(
        title="Attribute Value",
        overlaying="y",
        side="right",
        showgrid=False
    ),

    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.15,
            y=1
        ),
        dict(
            buttons=date_buttons,
            direction="down",
            showactive=True,
            x=0.35,
            y=1.15
        )
    ],

    template="plotly_white",
    height=650
)

fig.show()